In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm 

In [64]:
tqdm.pandas() 

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shrij\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shrij\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\shrij\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shrij\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [65]:
custom_stopwords = ['http', 'https', 'www', 'com', 'org', 'net', 'co']
stop_words.update(custom_stopwords)
lemmatizer = WordNetLemmatizer()

In [66]:
print("Loading dataset...")
df = pd.read_csv('../training_data.csv')
df1 = pd.read_csv('../testing_data.csv')

Loading dataset...


In [67]:
df

,text,label
0,"Subject: Minutes: Sales standup\nHi Riya,\n\nY...",0
1,"Subject: fw : winston debbie , this is an up...",0
2,online pharmacy visit our online store and sav...,2
3,http www spcm org journal spip php articleesca...,0
4,i have been having a weird non fatal problem l...,0
...,...,...
448154,as a dependable e supplier we know the reason ...,2
448155,cnn alerts physic_1950rsvpscom cnn alerts cust...,1
448156,"good morning ,\nwe we offer latest oem package...",2
448157,compañía americana esta solicitando distribuid...,2


In [68]:
df1

,text,label
0,"effective july 1 avails , beverly beaty will b...",0
1,john apologize tone e mail hope agree e mail c...,0
2,fax email mstyles favorites tagged any type of...,2
3,Subject: Your parcel is waiting (ID 633999)\nI...,1
4,don't waste the opportunity  anatrim  the ve...,2
...,...,...
112035,i will be out of the office thursday july 12 t...,0
112036,Subject: Exclusive deal on music subscription ...,1
112037,third order nanotechnologies achieves major te...,1
112038,"hi paliourg ,\nwe are your convenient , safe a...",2


In [69]:
print(df['label'].value_counts())

label
0                                                                                               210048
2                                                                                               123226
1                                                                                               110398
ham                                                                                               3887
spam                                                                                               595
 its termination would not  have such a phenomenal impact on the power situation .  however          1
 mr suresh prabhu                                                                                    1
{"mode":"full"                                                                                       1
Name: count, dtype: int64


In [70]:
df['text'] = df['text'].str.replace('Subject: ', '')
df1['text'] = df1['text'].str.replace('Subject: ', '')

In [71]:
df['label'] = df['label'].replace({'0': 'ham', '1': 'spam', '2': 'spam'})
print(df['label'].value_counts())
df = df[df['label'].isin(['ham', 'spam'])]
df['label'] = df['label'].replace({'ham': 0, 'spam': 1})

df1['label'] = df1['label'].replace({'0': 'ham', '1': 'spam', '2': 'spam'})
print(df1['label'].value_counts())
df1 = df1[df1['label'].isin(['ham', 'spam'])]
df1['label'] = df1['label'].replace({'ham': 0, 'spam': 1})

label
spam                                                                                            234219
ham                                                                                             213935
 its termination would not  have such a phenomenal impact on the power situation .  however          1
 mr suresh prabhu                                                                                    1
{"mode":"full"                                                                                       1
Name: count, dtype: int64
label
spam    58799
ham     53241
Name: count, dtype: int64


In [72]:
print(df['label'].value_counts())
print(df1['label'].value_counts())

label
1    234219
0    213935
Name: count, dtype: int64
label
1    58799
0    53241
Name: count, dtype: int64


In [73]:
print("Dropping duplicates...")
df = df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
df1 = df1.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)

Dropping duplicates...


In [74]:
print(df['label'].value_counts())
print(df1['label'].value_counts())

label
1    172306
0    154014
Name: count, dtype: int64
label
1    53668
0    48482
Name: count, dtype: int64


In [75]:
df['text'] = df['text'].fillna('')
df1['text'] = df1['text'].fillna('')

In [76]:
print("Calculating character counts...")
df['num_chars'] = df['text'].progress_apply(len)
df1['num_chars'] = df1['text'].progress_apply(len)

print("Calculating word counts...")
df['num_words'] = df['text'].progress_apply(lambda x: len(nltk.word_tokenize(str(x))))
df1['num_words'] = df1['text'].progress_apply(lambda x: len(nltk.word_tokenize(str(x))))

print("Calculating sentence counts...")
df['num_sentences'] = df['text'].progress_apply(lambda x: len(nltk.sent_tokenize(str(x))))
df1['num_sentences'] = df1['text'].progress_apply(lambda x: len(nltk.sent_tokenize(str(x))))

Calculating character counts...


100%|██████████████████████████████████████████████████████████████████████| 102150/102150 [00:00<00:00, 667030.17it/s]


Calculating word counts...


100%|█████████████████████████████████████████████████████████████████████████| 102150/102150 [01:47<00:00, 950.78it/s]


Calculating sentence counts...


100%|███████████████████████████████████████████████████████████████████████| 102150/102150 [00:08<00:00, 11999.04it/s]


In [77]:
def clean_text(text):
    # Added str() here just in case any completely blank cells slipped through
    # This prevents the float attribute error
    text = str(text).lower()

    url_regex = r'(https?:\/\/\S+|www\.\S+)'
    email_regex = r'[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}'
    num_regex = r'\d+'
    currency_regex = r'[$£€]'

    text = re.sub(url_regex, 'webaddr', text)
    text = re.sub(email_regex, 'emailaddr', text)
    text = re.sub(num_regex, 'num', text)
    text = re.sub(currency_regex, 'currency ', text)

    text = text.replace('escapenumber', 'num')

    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()

    # len(i) > 1 to filter out leftover tags such as x, e, etc. and seen in EDA
    clean_words = [lemmatizer.lemmatize(i) for i in words if i not in stop_words and len(i) > 1]

    return ' '.join(clean_words)


print("Running text preprocessing...")
df['clean_text'] = df['text'].progress_apply(clean_text)
df1['clean_text'] = df1['text'].progress_apply(clean_text)

Running text preprocessing...


100%|█████████████████████████████████████████████████████████████████████████| 102150/102150 [01:55<00:00, 884.43it/s]


In [78]:
print(df['clean_text'].isnull().sum())
print(df1['clean_text'].isnull().sum())

0
0


In [80]:
print(df['text'].value_counts().sum())
print(df1['text'].value_counts().sum())

326320
102150


In [81]:
df = df[df['clean_text'].str.strip() != '']
df1 = df1[df1['clean_text'].str.strip() != '']

In [82]:
print(df['text'].value_counts().sum())
print(df1['text'].value_counts().sum())

326222
102111


In [86]:
df = df[['text', 'clean_text', 'label', 'num_chars', 'num_words', 'num_sentences']]
df1 = df1[['text', 'clean_text', 'label', 'num_chars', 'num_words', 'num_sentences']]

In [87]:
print(df.columns.to_list())
print(df1.columns.to_list())

['text', 'clean_text', 'label', 'num_chars', 'num_words', 'num_sentences']
['text', 'clean_text', 'label', 'num_chars', 'num_words', 'num_sentences']


In [88]:
print("Saving preprocessed data to CSV...")
df.to_csv('../preprocessed_training_data.csv', index = False)
df1.to_csv('../preprocessed_testing_data.csv', index = False)

Saving preprocessed data to CSV...


In [2]:
!jupyter nbconvert --to script data_preprocessing.ipynb


[NbConvertApp] Converting notebook data_preprocessing.ipynb to script
[NbConvertApp] Writing 4252 bytes to data_preprocessing.py
